# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Schema URL:**  
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print summary
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets, fields, and their `@id` identifiers.

In [ ]:
# List record sets and fields using '@id'
record_sets = []
for rs in metadata.record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    print(f"  Description: {rs.get('description', 'N/A')}")
    print("  Fields:")
    if 'fields' in rs:
        for f in rs['fields']:
            print(f"    - Field @id: {f['@id']} | Name: {f.get('name', 'N/A')} | DataType: {f.get('dataType', 'N/A')}")
    record_sets.append(rs['@id'])
print(f"\nDiscovered record sets: {record_sets}")

In [ ]:
# Print records from each record set using `@id`
for record_set_id in record_sets:
    print(f"\nSample records from RecordSet {record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(json.dumps(record, indent=2))
        if i >= 2:
            break

## 3. Data Extraction
Load data from each record set into Pandas DataFrames. Use the record set and field `@id`s discovered above.

In [ ]:
# Extract dataframes for each record set using their @id
dataframes = dict()

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"\nColumns in record set {rs_id}: {dataframes[rs_id].columns.tolist()}")
    print(dataframes[rs_id].head(3))

# For further analysis, select the main tabular record set (usually the principal clinical table)
# Replace with the correct @id if needed:
main_record_set_id = record_sets[0]
main_df = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Apply basic processing: filtering, normalization, grouping, referencing all entities and columns by `@id`.

In [ ]:
# Identify numeric fields and group fields by their `@id`
numeric_fields = []
group_fields = []
fields_meta = None
for rs in metadata.record_sets:
    if rs['@id'] == main_record_set_id:
        fields_meta = rs.get('fields', [])
        break
if fields_meta is not None:
    for f in fields_meta:
        # heuristically pick possible numeric fields, e.g. Age, Interval, etc.
        if f.get('dataType','').lower() in ['integer','float','number','schema:integer','schema:float','schema:number']:
            numeric_fields.append(f['@id'])
        # Group fields (categorical), such as anatomical location or sex
        if f.get('dataType','').lower() in ['text','string','schema:text']:
            group_fields.append(f['@id'])
print(f"Numeric fields (@id): {numeric_fields}")
print(f"Group fields (@id): {group_fields}")

# Choose one numeric field and one group field for EDA
if len(numeric_fields) > 0:
    numeric_field_id = numeric_fields[0]
else:
    numeric_field_id = main_df.select_dtypes(include=['number']).columns[0]
if len(group_fields) > 0:
    group_field_id = group_fields[0]
else:
    group_field_id = main_df.select_dtypes(exclude=['number']).columns[0]

# Filtering and normalization
threshold = 10
if numeric_field_id in main_df.columns:
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head(3))
    # Normalize field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))

    # Optional: group by group field
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head(3))

## 5. Visualization
Visualize main numeric (`@id`) and categorical (`@id`) fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for the numeric field
if numeric_field_id in main_df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(main_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Bar plot grouped by group_field
if group_field_id in main_df.columns:
    plt.figure(figsize=(8, 4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=main_df)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR^2 dataset using the `mlcroissant` library. Key steps included referencing all schema entities by their `@id`, identifying main fields for analysis, filtering and normalizing clinical data, and visualizing core attributes. This approach ensures reproducibility and FAIR principles in biomedical data exploration.